# Deep Learning 077 — Multi-Head Attention

Companion notebook to the lesson. Multi-head attention is self-attention run several times
in parallel. There is no new mathematics in it, which makes it the right place to check the
two claims everyone repeats:

| Claim | What we measure |
|---|---|
| "multiple perspectives for the price of one" | **exact** — 786,432 either way |
| one head captures only one perspective | 1 head MSE **0.4226**, 2 heads **0.0000**, same parameters |
| different heads learn different roles | head 0 → 0.9943 on its target, head 1 → 0.9938 on the other |
| `W_o` mixes the perspectives | **it does not** — the ablation comes out flat, and the real reason is different |
| eight small heads ≈ one big head | **no** — `rank(QK^T) ≤ d_k`, and 10.6% is unreachable |

`numpy` for the arithmetic, `torch` for the trained part. The training cells take a couple
of minutes.

In [ ]:
import numpy as np

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(0)

## Part A — The split costs exactly nothing

Eight heads sounds like eight times the work. It is not, because `d_model` is **divided
among** the heads rather than duplicated: each head gets `d_k = d_model / h`.

In [ ]:
D_MODEL, H = 512, 8
D_K = D_MODEL // H
print(f"d_model = {D_MODEL}, h = {H}, so d_k = {D_K}\n")

one_big = 3 * D_MODEL * D_MODEL
per_head = 3 * D_MODEL * D_K
all_heads = H * per_head
W_O = D_MODEL * D_MODEL

print(f"{'one 512-dim head:  3 x 512 x 512':<36}{one_big:>10,}")
print(f"{'one 64-dim head:   3 x 512 x 64':<36}{per_head:>10,}")
print(f"{'eight of them':<36}{all_heads:>10,}")
assert all_heads == one_big                    # <- equal, not approximately
print(f"{'':<36}{'':>10}   <- identical\n")
print(f"{'output projection W_o: 512 x 512':<36}{W_O:>10,}")
print(f"{'the whole block':<36}{all_heads + W_O:>10,}")
print(f"\nmulti-head costs {(all_heads + W_O) / one_big:.2f}x a single 512-dim head, "
      f"and all of the extra is W_o")

In [ ]:
# the score arithmetic cancels the same way
n = 30
print(f"one 512-dim head : n^2 * d_model = {n * n * D_MODEL:>10,}")
print(f"8 heads of 64    : h * n^2 * d_k = {H * n * n * D_K:>10,}")
assert H * n * n * D_K == n * n * D_MODEL
print("\nEqual, because h * d_k IS d_model. The split is a reshape of the same work.")

The `h` cancels: `h × 3·d_model·(d_model/h) = 3·d_model²`. That is the whole content of
"multiple perspectives for the price of one", and it is an identity rather than an
approximation.

## Part B — The block, end to end

Watch the shapes. The last one is a **requirement**: the block's output is added back into
the stream its input came from, so it must match the input shape or the block cannot be
stacked.

In [ ]:
X = rng.normal(size=(2, D_MODEL))                      # "money bank"
Wq = rng.normal(size=(H, D_MODEL, D_K)) / np.sqrt(D_MODEL)
Wk = rng.normal(size=(H, D_MODEL, D_K)) / np.sqrt(D_MODEL)
Wv = rng.normal(size=(H, D_MODEL, D_K)) / np.sqrt(D_MODEL)
Wo = rng.normal(size=(D_MODEL, D_MODEL)) / np.sqrt(D_MODEL)

Q = np.einsum("nd,hdk->hnk", X, Wq)                    # (8, 2, 64) - one Q per head
K = np.einsum("nd,hdk->hnk", X, Wk)
V = np.einsum("nd,hdk->hnk", X, Wv)
A = softmax(Q @ K.transpose(0, 2, 1) / np.sqrt(D_K))   # (8, 2, 2) - EIGHT score tables
Z = A @ V                                              # (8, 2, 64)
cat = Z.transpose(1, 0, 2).reshape(2, D_MODEL)         # (2, 512) - in HEAD order
out = cat @ Wo                                         # (2, 512) - in model order

for name, arr in (("X", X), ("Q per head", Q), ("scores per head", A),
                  ("Z per head", Z), ("concatenated", cat), ("output", out)):
    print(f"{name:>18}  {str(arr.shape):>14}")
assert out.shape == X.shape

## Part C — Does a second head actually buy anything?

"Multiple perspectives" is a pleasant phrase, not evidence. Because the split holds the
parameter count fixed, the claim is directly testable: **same budget, more heads, a task
that needs two answers at once.**

Every token is `[tag A | tag B | payload]`. The query carries one A-tag and one B-tag and
no payload. The answer is the payload of the A-matching token **concatenated with** the
payload of the B-matching token — two retrievals, from two different places, in one step.

In [ ]:
import torch

N_TAG, N_PAY, SEQ = 6, 8, 7
D_TOK = 2 * N_TAG + N_PAY
D_TOT = 16                                   # total width, held FIXED across head counts
g = torch.Generator().manual_seed(1)

def distinct_tags(bs):
    # argsort of noise is a batch of random permutations
    return torch.rand(bs, N_TAG, generator=g).argsort(1)[:, :SEQ - 1]

def batch_of(bs):
    rows = torch.arange(bs)
    a = torch.zeros(bs, SEQ, dtype=torch.long)
    b = torch.zeros(bs, SEQ, dtype=torch.long)
    a[:, 1:] = distinct_tags(bs)
    b[:, 1:] = distinct_tags(bs)
    a[:, 0] = a[rows, 1 + torch.randint(0, SEQ - 1, (bs,), generator=g)]
    b[:, 0] = b[rows, 1 + torch.randint(0, SEQ - 1, (bs,), generator=g)]
    pay = torch.randn(bs, SEQ, N_PAY, generator=g)
    pay[:, 0] = 0.0                          # the query carries no payload
    x = torch.cat([torch.nn.functional.one_hot(a, N_TAG).float(),
                   torch.nn.functional.one_hot(b, N_TAG).float(), pay], -1)
    hit_a, hit_b = (a[:, 1:] == a[:, :1]), (b[:, 1:] == b[:, :1])
    target = torch.cat([(pay[:, 1:] * hit_a.unsqueeze(-1)).sum(1),
                        (pay[:, 1:] * hit_b.unsqueeze(-1)).sum(1)], -1)
    return x, target, hit_a.float().argmax(1), hit_b.float().argmax(1)

x, t, ia, ib = batch_of(3)
print("token shape", x.shape, " target shape", t.shape)
print("first example: A-token is index", ia[0].item(), ", B-token is index", ib[0].item())

g = torch.Generator().manual_seed(1)         # reset, so this peek does not shift the
                                             # stream and the numbers below match the lesson

In [ ]:
ROT = torch.linalg.qr(torch.randn(2 * N_PAY, 2 * N_PAY,
                                  generator=torch.Generator().manual_seed(3)))[0]

def train(heads, steps=3000, bs=256, use_wo=True, rotate=False, seed=7):
    torch.manual_seed(seed)
    d_h = D_TOT // heads
    Wq = torch.randn(heads, D_TOK, d_h, requires_grad=True)
    Wk = torch.randn(heads, D_TOK, d_h, requires_grad=True)
    Wv = torch.randn(heads, D_TOK, d_h, requires_grad=True)
    Wo = torch.randn(D_TOT, 2 * N_PAY, requires_grad=True)
    with torch.no_grad():
        for p in (Wq, Wk, Wv, Wo):
            p /= np.sqrt(D_TOK)
    params = [Wq, Wk, Wv] + ([Wo] if use_wo else [])
    opt = torch.optim.Adam(params, lr=0.01)

    def forward(x):
        q = torch.einsum("bd,hdk->bhk", x[:, 0], Wq)
        k = torch.einsum("bsd,hdk->bhsk", x[:, 1:], Wk)
        v = torch.einsum("bsd,hdk->bhsk", x[:, 1:], Wv)
        att = torch.softmax(torch.einsum("bhk,bhsk->bhs", q, k) / np.sqrt(d_h), -1)
        z = torch.einsum("bhs,bhsk->bhk", att, v)
        cat = z.reshape(z.shape[0], -1)                    # concatenate the heads
        return (cat @ Wo if use_wo else cat), att

    prep = (lambda t: t @ ROT) if rotate else (lambda t: t)
    for _ in range(steps):
        x, target, _, _ = batch_of(bs)
        out, _ = forward(x)
        loss = (out - prep(target)).pow(2).mean()
        opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        mses, am, bm = [], [], []
        for _ in range(20):
            x, target, ia, ib = batch_of(512)
            out, att = forward(x)
            mses.append((out - prep(target)).pow(2).mean().item())
            rows = torch.arange(len(ia))
            am.append(att[rows, 0, ia].mean().item())
            if heads > 1:
                bm.append(att[rows, 1, ib].mean().item())
    return (float(np.mean(mses)), sum(p.numel() for p in params),
            float(np.mean(am)), float(np.mean(bm)) if bm else float("nan"), forward)

In [ ]:
floor = float(np.mean([batch_of(512)[1].pow(2).mean().item() for _ in range(20)]))
print(f"predicting zeros scores MSE {floor:.4f} - the do-nothing floor\n")
print(f"{'heads':>7}{'d_h':>6}{'params':>9}{'MSE':>10}{'head 0 -> A':>14}{'head 1 -> B':>14}")
fwds = {}
for h in (1, 2, 4, 8):
    mse, npar, am, bm, fwd = train(h)
    fwds[h] = fwd
    print(f"{h:>7}{D_TOT // h:>6}{npar:>9}{mse:>10.4f}{am:>14.3f}"
          f"{'-' if np.isnan(bm) else f'{bm:.3f}':>14}")

**Every row has 1,216 parameters.** The single head is not smaller, not under-trained, and
still cannot do the task.

The reason is structural. A softmax row is **one convex combination**, and every value it
mixes came through the **same `W_v`**. So one head's output is symmetric in the tokens it
averaged — it can produce a blend of the A-token and the B-token, but it has no way to say
*these eight numbers came from that token and those eight from the other*. Two independent
softmaxes writing into two disjoint slices can.

That is the ambiguous-sentence problem — *"the man saw the astronomer with a telescope"* —
in a form that produces a number.

## Part D — Nobody assigns the heads their jobs

In [ ]:
with torch.no_grad():
    x, target, ia, ib = batch_of(4096)
    _, att = fwds[2](x)
    rows = torch.arange(len(ia))
    print("2-head model, 4096 held-out queries:\n")
    print(f"{'':>8}{'weight on A-token':>20}{'weight on B-token':>20}")
    for h in (0, 1):
        print(f"{'head ' + str(h):>8}{att[rows, h, ia].mean():>20.4f}"
              f"{att[rows, h, ib].mean():>20.4f}")
    print(f"\nuniform attention over {SEQ - 1} candidates would be {1 / (SEQ - 1):.4f}")

There is no diversity penalty in the loss, no orthogonality constraint, nothing that
rewards head 0 for taking the A-relation. **The loss needed two answers, the architecture
offered two independent ways to look, and gradient descent assigned one to each** — because
any solution where both heads chase the same relation leaves the other unanswered.

Worth knowing that this is *emergent*, not guaranteed. Real models do end up with redundant
heads, and a good deal of interpretability work is about finding out which.

## Part E — What `W_o` is really for

The standard story is that `W_o` weighs the perspectives and decides how important each one
is. That is testable, so test it.

In [ ]:
print(f"{'target layout':<36}{'with W_o':>11}{'concat only':>14}")
for rotate, label in ((False, "head-aligned (as built above)"),
                      (True, "rotated into a random basis")):
    a = train(2, use_wo=True, rotate=rotate)[0]
    b = train(2, use_wo=False, rotate=rotate)[0]
    print(f"{label:<36}{a:>11.4f}{b:>14.4f}")

**Row one is a null result and it is the informative half.** The task was built so the
answer is literally "head 0's slice, then head 1's" — concatenation already produces exactly
that, and `W_o` has nothing to do. So the "weighs the perspectives" story is not what the
measurement supports.

Row two is the situation a real block is in: its output is added straight back into a
residual stream whose coordinates it does not own, so the answer is in *some other basis*.
Now concatenation cannot get there.

**`W_o` is not adding power. It is changing basis.** Concatenation hands you an answer in
*head coordinates*, where slot `h*d_h` onward belongs to head `h` and nothing else may write
there; the rest of the model needs it in *model coordinates*. That costs 262,144 parameters
— a third of the block.

## Part F — Why the heads are narrow, which is not only about cost

`Q @ K.T` is an `(n × d_k)` times a `(d_k × n)`, so **its rank is at most `d_k`**. If the
sentence is longer than the head is wide, a single head cannot express an arbitrary pattern
of who-attends-to-whom, no matter how it is trained.

In [ ]:
N_TOK = 128
S = rng.normal(size=(N_TOK, N_TOK))                 # some target attention pattern
sv = np.linalg.svd(S, compute_uv=False)
energy = (sv ** 2).cumsum() / (sv ** 2).sum()

print(f"a random {N_TOK} x {N_TOK} score matrix, best rank-r approximation:\n")
print(f"{'d_k = r':>9}{'energy reachable':>19}{'out of reach':>15}")
for r in (8, 16, 32, 64, 128):
    print(f"{r:>9}{energy[r - 1]:>19.3f}{1 - energy[r - 1]:>14.1%}")

At `d_k = 64` over 128 tokens, **10.6% of the pattern is unreachable** by any single head.
Eight heads do not fix that by being bigger — each one is just as limited. They fix it by
being *eight*, and by `W_o` being free to recombine them.

Which reframes the design. Splitting 512 into 8 × 64 is usually sold as a pure saving. It is
also a **rank constraint accepted on purpose**: eight cheap low-rank views, recombined, in
place of one expensive full-rank one. And it is plausibly *why* the specialisation in Part D
happens at all — a head that could represent everything would feel no pressure to represent
one thing well.

## Try it yourself

1. Re-run Part C with `D_TOT = 32`. Does one head succeed now? Predict first — is the
   failure about capacity or about structure?
2. Build a task needing **three** simultaneous retrievals. At what head count does it
   become solvable, and does it match your prediction?
3. In Part D, initialise both heads identically (same seed, same values). Do they still
   specialise, or do they stay tied?
4. Compute the rank table for `n = 32` tokens instead of 128. At `d_k = 64`, how much is
   out of reach — and what does that say about short sentences?